# 0. Getting started

These notebooks reproduce each section and each figure of Paper I, the cosmology and halo layers of `ggah_mod` and the constraints they give.
Each notebook follows one section: the equations, the calls that compute them step by step, the figure of the paper, a check of its numbers against the text, and exercises.
This one sets up the environment and introduces the two objects every later notebook uses: the cosmology container and the backend that says how to compute with it.

| Notebook | Paper I | Figures |
|---|---|---|
| 01 Cosmology and background | Secs. 2.1–2.2 | distance gradients |
| 02 Power spectrum | Sec. 2.3 | $P(k)$ gradients |
| 03 Amplitude and growth | Secs. 2.4–2.5 | amplitude gradients |
| 04 Variance and peak height | Sec. 3.1 | variance |
| 05 Abundance and bias | Secs. 3.2–3.3 | abundance |
| 06 Concentration and profiles | Secs. 3.4–3.7 | concentration, profile |
| 07 Distance constraints | Sec. 4.1 | distance Fisher |
| 08 Full shape | Sec. 4.2 | $P(k)$ Fisher, with BAO and BBN |
| 09 Halo counts | Sec. 4.3 | mass-function Fisher |
| 10 Halo-layer requirements | Sec. 4.4 | halo requirements |
| 11 A joint halo emulator | Secs. 4.5–4.6 | suite errors |
| A Emulator benchmark | App. A | emulator benchmark |
| B Code comparisons | App. B | comparison tables |

Notebooks 01–09 compute everything from a `pip install`.
Notebooks 10, 11, A and B also read precomputed results from the tutorial data bundle (see the end of this notebook).

## Installation

```
pip install "ggah_mod[tutorials]"
```

installs `ggah_mod`, `emu_pk`, `matplotlib`, `pooch` and `jupyter`.
CLASS (`pip install "ggah_mod[reference]"`) is needed only for the cells that compare against it, which are skipped otherwise.
The notebooks import `_helpers.py` from their own directory, so run them from `tutorials/`.

`ggah_mod` assumes double precision.
JAX defaults to single precision, so every notebook starts by switching it on, before the first `ggah_mod` import.

In [1]:
import jax
jax.config.update("jax_enable_x64", True)   # before the first ggah_mod import

from importlib.metadata import version

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import _helpers as T
T.paper_style()
for name in ("ggah_mod", "emu_pk", "jax", "numpy"):
    print(f"{name:9s} {version(name)}")

ggah_mod  1.7.0
emu_pk    2.1.1
jax       0.10.2
numpy     2.4.6


## The cosmology container

A `Cosmology` holds eleven fields.
Ten are differentiable leaves of a JAX pytree, so `jax.grad` of anything downstream flows through them; the eleventh, the neutrino ordering, is static.
`PLANCK18` is the default of the paper (Eq. 1): the *Planck* 2018 TT,TE,EE+lowE+lensing means at the minimal neutrino mass.

In [2]:
from ggah_mod.cosmology import PLANCK18, Cosmology

print(PLANCK18)
c = PLANCK18.replace(Omega_m=0.30, w0=-0.9)   # a new container; PLANCK18 is unchanged
print(c.Omega_m, c.w0, PLANCK18.Omega_m)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Cosmology(Omega_m=0.3153, Omega_b=0.0493, h=0.6736, n_s=0.9649, ln10A_s=3.044, sum_mnu=0.06, w0=-1.0, wa=0.0, Omega_k=0.0, T_cmb=2.7255, nu_hierarchy='degenerate')
0.3 -0.9 0.3153


$\sigma_8$ and $S_8$ are outputs of the power spectrum, not inputs; asking for one raises.
`ln10A_s_for_sigma8` returns the amplitude that gives a requested $\sigma_8$ (notebook 03).

In [3]:
try:
    Cosmology.create(sigma8=0.8)
except TypeError as err:
    print(err)

'sigma8' is not an input to this package.  The amplitude is 'ln10A_s' and sigma8/S8 are derived from the spectrum -- see ggah_mod.cosmology.amplitude.sigma8, .s8.  A cosmology carrying both an amplitude and a sigma8 is over-determined, and which one wins used to depend on which module you were in.  To start from a sigma8, solve for the amplitude once, up front, with amplitude.ln10A_s_for_sigma8().


## Two flavours

A backend says how to compute: which power spectrum, how fine the grids, which halo fits.
Two are provided.
`ACCURATE` is the fitting path: a Boltzmann solver (CLASS by default) on fine grids.
`DIFFERENTIABLE` is the forecasting path: the `emu_pk` emulator, the only differentiable backend, on coarser grids, traced end to end.

In [4]:
from ggah_mod import ACCURATE, DIFFERENTIABLE

print(ACCURATE)
print(DIFFERENTIABLE)

accurate(pk=class, n_k=1024, n_m=512, n_gl=200, mdef=200m, hmf=tinker08, bias=tinker10, cm=bhattacharya13, traced=False)
differentiable(pk=emu_pk, n_k=512, n_m=256, n_gl=64, mdef=200m, hmf=tinker08, bias=tinker10, cm=bhattacharya13, traced=True)


Every notebook uses `DIFFERENTIABLE`, since the paper's figures are derivatives and forecasts.
A gradient through a Boltzmann solver raises `NotDifferentiableError` rather than returning zero.

In [5]:
from ggah_mod.cosmology import make_pk, sigma8
from ggah_mod.cosmology.power import NotDifferentiableError

k = np.logspace(-4, np.log10(50.0), 512)
pk = make_pk("emu_pk")


def sigma8_of(omega_m, pk):
    return sigma8(pk.pk(k, 0.0, PLANCK18.replace(Omega_m=omega_m)), k)


print("sigma8 =", float(sigma8_of(PLANCK18.Omega_m, pk)))
print("d sigma8 / d Omega_m =", float(jax.grad(sigma8_of)(PLANCK18.Omega_m, pk)))

try:
    jax.grad(sigma8_of)(PLANCK18.Omega_m, make_pk("class"))
except NotDifferentiableError as err:
    print("CLASS:", str(err).splitlines()[0])
except ImportError:
    print("classy is not installed; the CLASS backend is optional")

sigma8 = 0.8132492762349499


d sigma8 / d Omega_m = 1.9484815647880902
CLASS: ClassPk is a Boltzmann solver called from Python and cannot be traced, differentiated or compiled; it was handed traced values for ['Omega_m'].  Use the DIFFERENTIABLE flavour (`make_pk('emu_pk')`, `ggah_mod.DIFFERENTIABLE`) under jax.grad, jax.jit or jax.vmap, and ACCURATE for point evaluations, or take finite differences of ACCURATE outside JAX.


## The tutorial data

Notebooks 07–11, A and B read files from a data bundle on Zenodo: published DESI posteriors drawn over the forecasts, and results that took cluster time to compute.
`_helpers.data_path(name)` returns the local path of a file.
It looks in the directory named by `GGAH_TUTORIALS_DATA`, then in `~/.cache/ggah_mod/tutorials`, and downloads the bundle once otherwise.
Each file is checked against its SHA-256 in `data_registry.txt`, so a notebook reads exactly the file the paper was made from.

In [6]:
for name in list(T._registry())[:4]:
    print(name)
print("...", len(T._registry()), "files")

desi/desi_dr2_contours.json
desi/desi_dr1_fs_contours.json
halo_req/counts/results.json
halo_req/counts/kernels.npz
... 21 files


## Run time

Each notebook has a `FAST` switch, set from the environment variable `GGAH_TUTORIALS_FAST` (default on).
With `FAST` on, grids are reduced so that a notebook runs in a few minutes on a laptop CPU; with it off, the notebook reproduces the figure of the paper exactly.
The first call of a function compiles it, so the second is the one to time.

In [7]:
import time

f = jax.jit(lambda om: sigma8_of(om, pk))
for label in ("first call (compiles)", "second call"):
    t0 = time.perf_counter()
    f(0.31).block_until_ready()
    print(f"{label:22s} {1e3 * (time.perf_counter() - t0):8.2f} ms")

first call (compiles)    175.96 ms
second call                0.65 ms


## Exercises

1. Build a cosmology with `nu_hierarchy="inverted"` and `sum_mnu=0.06`, read the error, and find the smallest sum it accepts.
2. Compute $\partial\sigma_8/\partial h$ with `jax.grad`, and check it against a central finite difference with a step of $10^{-4}$.
3. Time `sigma8_of` with and without `jax.jit` for a vector of 100 values of $\Omega_{\rm m}$, using `jax.vmap`.